# Lab 05: OpenTelemetry Setup

**Goal:** Learn how to configure OpenTelemetry SDK for Python —
TracerProvider, MeterProvider, and exporters.

**What you'll learn:**
- OpenTelemetry architecture and OTLP protocols
- Python OTel packages and their purposes
- TracerProvider setup with BatchSpanProcessor
- OTel Collector configuration (receivers, processors, exporters)
- Environment-variable-based SDK configuration

In [ ]:
import os
import shutil
import textwrap

WORKDIR = "/tmp/k8s-lab-10-05"

if os.path.exists(WORKDIR):
    shutil.rmtree(WORKDIR)
os.makedirs(WORKDIR, exist_ok=True)
print(f"Working directory: {WORKDIR}")

## Step 1: OpenTelemetry Architecture

OpenTelemetry (OTel) = vendor-neutral observability framework

In [ ]:
print("Architecture:")
print("  App (SDK) ──OTLP──> OTel Collector ──> Backends")
print("                        │")
print("                   ┌────┴────────────────────────────┐")
print("                   │  receivers → processors →    │")
print("                   │              exporters       │")
print("                   └─────────────────────────────────┘")
print("                        │         │         │")
print("                    LangFuse  LangFuse   LangFuse")
print("                    (traces)  (metrics)  (logs)")
print()
print("OTLP Protocols:")
print("  gRPC   port 4317 — binary protobuf (default, fastest)")
print("  HTTP   port 4318 — HTTP/protobuf (firewall-friendly)")

## Step 2: Python OTel Packages

In [ ]:
packages = [
    ("opentelemetry-api",                       "Core API (tracing, metrics)"),
    ("opentelemetry-sdk",                       "SDK implementation"),
    ("opentelemetry-exporter-otlp",             "OTLP exporter (gRPC + HTTP)"),
    ("opentelemetry-instrumentation-fastapi",   "Auto-instrument FastAPI"),
    ("opentelemetry-instrumentation-requests",  "Auto-instrument HTTP requests"),
    ("opentelemetry-instrumentation-redis",     "Auto-instrument Redis calls"),
    ("opentelemetry-instrumentation-logging",   "Inject trace_id into logs"),
]

print(f"{'Package':<50} {'Purpose'}")
print(f"{'-'*80}")
for pkg, purpose in packages:
    print(f"{pkg:<50} {purpose}")

print("\nQuick install:")
print("  pip install opentelemetry-distro")
print("  opentelemetry-bootstrap -a install")

## Step 3: TracerProvider Setup

The TracerProvider is the core of OTel tracing.

In [ ]:
setup_code = textwrap.dedent("""\
    from opentelemetry import trace
    from opentelemetry.sdk.trace import TracerProvider
    from opentelemetry.sdk.trace.export import BatchSpanProcessor
    from opentelemetry.exporter.otlp.proto.grpc.trace_exporter import OTLPSpanExporter
    from opentelemetry.sdk.resources import Resource

    resource = Resource.create({
        "service.name": "agent-api",
        "service.version": "2.0.0",
        "deployment.environment": "production",
    })

    provider = TracerProvider(resource=resource)
    processor = BatchSpanProcessor(
        OTLPSpanExporter(endpoint="http://otel-collector:4317")
    )
    provider.add_span_processor(processor)
    trace.set_tracer_provider(provider)

    tracer = trace.get_tracer("agent.api")
""")

for line in setup_code.strip().split("\n"):
    print(f"  {line}")

## TODO 1: OTel Collector Configuration

Create an OTel Collector configuration with:
- Receiver: OTLP (gRPC on 4317, HTTP on 4318)
- Processor: batch (timeout 5s, batch_size 1000)
- Processor: memory_limiter (limit_mib 512)
- Exporter: otlp/langfuse (endpoint langfuse:4317, insecure)
- Pipeline traces: otlp -> memory_limiter,batch -> otlp/langfuse
- Pipeline metrics: otlp -> memory_limiter,batch -> otlp/langfuse

In [ ]:
# TODO: OTel Collector Configuration
# Include: receivers, processors, exporters, service.pipelines

# todo1_yaml = textwrap.dedent("""\
#     receivers:
#       otlp:
#         protocols:
#           grpc:
#             endpoint: 0.0.0.0:4317
#           http:
#             endpoint: 0.0.0.0:4318
#
#     processors:
#       batch:
#         timeout: 5s
#         send_batch_size: 1000
#       memory_limiter:
#         limit_mib: 512
#
#     exporters:
#       otlp/langfuse:
#         endpoint: langfuse:4317
#         tls:
#           insecure: true
#
#     service:
#       pipelines:
#         traces:
#           receivers: [otlp]
#           processors: [memory_limiter, batch]
#           exporters: [otlp/langfuse]
#         metrics:
#           receivers: [otlp]
#           processors: [memory_limiter, batch]
#           exporters: [otlp/langfuse]
# """)

todo1_yaml = textwrap.dedent("""\
    # TODO: OTel Collector Configuration
    # Include: receivers, processors, exporters, service.pipelines

""")

with open(os.path.join(WORKDIR, "otel-collector-config.yaml"), "w") as f:
    f.write(todo1_yaml)

In [ ]:
collector_checks = [
    ("Has receivers section",       "receivers:" in todo1_yaml),
    ("Has OTLP receiver",           "otlp:" in todo1_yaml),
    ("Has gRPC port 4317",          "4317" in todo1_yaml),
    ("Has HTTP port 4318",          "4318" in todo1_yaml),
    ("Has processors section",      "processors:" in todo1_yaml),
    ("Has batch processor",         "batch:" in todo1_yaml),
    ("Has memory_limiter",          "memory_limiter:" in todo1_yaml),
    ("Has exporters section",       "exporters:" in todo1_yaml),
    ("Has langfuse exporter",       "langfuse" in todo1_yaml),
    ("Has service.pipelines",       "pipelines:" in todo1_yaml),
    ("Has traces pipeline",         "traces:" in todo1_yaml),
    ("Has metrics pipeline",        "metrics:" in todo1_yaml),
]

score1 = sum(1 for _, ok in collector_checks if ok)
print(f"Validating Collector Config ({score1}/{len(collector_checks)}):\n")
for name, ok in collector_checks:
    print(f"  [{'PASS' if ok else 'FAIL'}] {name}")

## TODO 2: Python Launch Script with OTel Environment

Create a Python launch script that configures OTel env vars
and starts a uvicorn server (no Docker required):
- Set OTEL_SERVICE_NAME: agent-api
- Set OTEL_EXPORTER_OTLP_ENDPOINT: http://localhost:4317
- Set OTEL_RESOURCE_ATTRIBUTES: deployment.environment=production
- Set OTEL_TRACES_SAMPLER: parentbased_traceidratio
- Set OTEL_TRACES_SAMPLER_ARG: 0.1
- Launch uvicorn on host 0.0.0.0, port 8000

In [ ]:
# TODO: Python launch script with OTel environment variables
# Set os.environ for each OTEL_* variable, then configure uvicorn

# todo2_code = textwrap.dedent("""\
#     import os
#     import uvicorn
#
#     # OTel environment configuration
#     os.environ["OTEL_SERVICE_NAME"] = "agent-api"
#     os.environ["OTEL_EXPORTER_OTLP_ENDPOINT"] = "http://localhost:4317"
#     os.environ["OTEL_RESOURCE_ATTRIBUTES"] = "deployment.environment=production"
#     os.environ["OTEL_TRACES_SAMPLER"] = "parentbased_traceidratio"
#     os.environ["OTEL_TRACES_SAMPLER_ARG"] = "0.1"
#
#     # Launch uvicorn on port 8000
#     if __name__ == "__main__":
#         uvicorn.run(
#             "app:app",
#             host="0.0.0.0",
#             port=8000,
#             log_level="info",
#         )
# """)

todo2_code = textwrap.dedent("""\
    # TODO: Python launch script with OTel environment variables
    # Set os.environ for each OTEL_* variable, then configure uvicorn

""")

with open(os.path.join(WORKDIR, "launch_agent_otel.py"), "w") as f:
    f.write(todo2_code)

In [ ]:
deploy_checks = [
    ("Has os.environ usage",        "os.environ" in todo2_code),
    ("Has OTEL_SERVICE_NAME",       "OTEL_SERVICE_NAME" in todo2_code),
    ("Has OTEL_EXPORTER_OTLP_ENDPOINT", "OTEL_EXPORTER_OTLP_ENDPOINT" in todo2_code),
    ("Has localhost endpoint",      "localhost" in todo2_code),
    ("Has OTEL_RESOURCE_ATTRIBUTES", "OTEL_RESOURCE_ATTRIBUTES" in todo2_code),
    ("Has OTEL_TRACES_SAMPLER",     "OTEL_TRACES_SAMPLER" in todo2_code),
    ("Has sampling ratio 0.1",      "0.1" in todo2_code),
    ("Has uvicorn config",          "uvicorn" in todo2_code),
]

score2 = sum(1 for _, ok in deploy_checks if ok)
print(f"Validating Launch Script ({score2}/{len(deploy_checks)}):\n")
for name, ok in deploy_checks:
    print(f"  [{'PASS' if ok else 'FAIL'}] {name}")

## Summary

Key concepts:
1. OTel SDK -> OTLP -> Collector -> Backends (LangFuse)
2. TracerProvider + BatchSpanProcessor + OTLPSpanExporter
3. Resource identifies service (name, version, environment)
4. OTEL_* env vars configure SDK without code changes

In [ ]:
print(f"TODO 1: {score1}/{len(collector_checks)} collector config checks passed")
print(f"TODO 2: {score2}/{len(deploy_checks)} launch script checks passed")
print(f"\nFiles generated in {WORKDIR}/")

## Key Takeaways

- **OTel architecture:** App (SDK) sends data via OTLP to a Collector, which routes to backends like LangFuse
- **TracerProvider:** Core tracing component configured with Resource, BatchSpanProcessor, and OTLPSpanExporter
- **Collector config:** Defines receivers (OTLP gRPC/HTTP), processors (batch, memory_limiter), and exporters per pipeline
- **Environment variables:** OTEL_* env vars allow SDK configuration without code changes (service name, endpoint, sampling)
- **Python packages:** `opentelemetry-distro` + `opentelemetry-bootstrap` for quick setup of auto-instrumentation